In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA bronze;
SET TIME ZONE 'UTC';

In [0]:
COPY INTO bronze.raw_characters
FROM (
  SELECT
    sha1(concat_ws('|', _metadata.file_path, cast(_metadata.file_modification_time AS STRING))) AS raw_characters_id,
    from_json(
      to_json(character),
      'STRUCT<
        character: STRUCT<
          name: STRING,
          traded: BOOLEAN,
          deletion_date: STRING,
          former_names: ARRAY<STRING>,
          title: STRING,
          unlocked_titles: INT,
          sex: STRING,
          vocation: STRING,
          level: INT,
          achievement_points: INT,
          world: STRING,
          former_worlds: ARRAY<STRING>,
          residence: STRING,
          married_to: STRING,
          houses: ARRAY<STRUCT<
            houseid: INT,
            name: STRING,
            town: STRING,
            paid: STRING
          >>,
          guild: STRUCT<
            name: STRING,
            rank: STRING
          >,
          last_login: STRING,
          comment: STRING,
          position: STRING,
          account_status: STRING
        >,
        account_badges: ARRAY<STRUCT<
          name: STRING,
          description: STRING,
          icon_url: STRING
        >>,
        achievements: ARRAY<STRUCT<
          grade: INT,
          name: STRING,
          secret: BOOLEAN
        >>,
        deaths_truncated: BOOLEAN,
        deaths: ARRAY<STRUCT<
          time: STRING,
          level: INT,
          reason: STRING,
          assists: ARRAY<STRUCT<
            name: STRING,
            player: BOOLEAN,
            traded: BOOLEAN,
            summon: STRING
          >>,
          killers: ARRAY<STRUCT<
            name: STRING,
            player: BOOLEAN,
            traded: BOOLEAN,
            summon: STRING
          >>
        >>,
        account_information: STRUCT<
          position: STRING,
          loyalty_title: STRING,
          created: STRING
        >,
        other_characters: ARRAY<STRUCT<
          name: STRING,
          position: STRING,
          world: STRING,
          status: STRING,
          main: BOOLEAN,
          traded: BOOLEAN,
          deleted: BOOLEAN
        >>
      >'
    ) AS character,
    from_json(
      to_json(information),
      'STRUCT<
        api: STRUCT<
          version: INT,
          release: STRING,
          commit: STRING
        >,
        timestamp: STRING,
        tibia_urls: ARRAY<STRING>,
        status: STRUCT<
          error: INT,
          http_code: INT,
          message: STRING
        >
      >'
    ) AS information,
    cast(regexp_extract(_metadata.file_path, '([0-9]{4}-[0-9]{2}-[0-9]{2})', 1) AS DATE) AS source_file_date,
    _metadata.file_path AS source_file_path,
    _metadata.file_modification_time AS source_file_modified_at,
    current_timestamp() AS ingested_at
  FROM '/Volumes/tibia_analytics/bronze/raw/characters/*'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false');

In [0]:
COPY INTO bronze.raw_characters_attempts
FROM (
  SELECT
    character_id,
    name,
    world,
    status,
    cast(regexp_extract(_metadata.file_path, '([0-9]{4}-[0-9]{2}-[0-9]{2})', 1) AS DATE) AS source_file_date,
    _metadata.file_path AS source_file_path,
    _metadata.file_modification_time AS source_file_modified_at,
    current_timestamp() AS ingested_at
  FROM '/Volumes/tibia_analytics/bronze/raw/attempts/characters/*'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false');